## Install Libraries

Runs the **dense retrieval baseline**: encode each gold mention with the
SapBERT checkpoint, scan the KB vectors built by `build_KB.ipynb`, and take the
top-k concepts. There is **no LLM feedback, no fusion and no re-ranking model**
— the Top-1 candidate is the prediction.

This notebook uses only `source/common/` (encoder, vector index, loaders,
metrics); it does not import the GRF pipeline. The output uses the same
candidate-file format as `retrieval.ipynb`, so `e2e.ipynb` can consume it
unchanged.

`tqdm` is installed explicitly for the progress bars.

In [ ]:
%pip install "tqdm>=4.66" "sentence-transformers>=2.7"

## Import Libraries

Third-party imports only. The shared `common` package lives inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.

In [ ]:
from __future__ import annotations

import gc
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

## Constants and Configuration

All paths and retrieval parameters live here. The Git repository and branch are
preconfigured; set `DRIVE_DATA_DIR` once. The baseline needs only the KB
artifact and the gold data — the KB CSV is not read, because no concept names
or LLM feedback are used.

`ENCODER_MODEL` is a HuggingFace id or a local model directory (for example a
folder on Drive). It must match the checkpoint recorded in the artifact.

In [ ]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive inputs and output (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
DRIVE_DATA_DIR = DRIVE_ROOT / "REPLACE_ME"                      # TODO: set once
KB_ARTIFACT_PATH = DRIVE_DATA_DIR / "grf_kb.npz"                # from build_KB.ipynb
GOLD_DATA_PATH = DRIVE_DATA_DIR / "test.jsonl"                  # gold BEL JSONL
BASELINE_CANDIDATES_PATH = DRIVE_DATA_DIR / "grf_baseline_candidates.jsonl"

# ---------------------------------------------------------------------------
# Encoder (HuggingFace id or a local directory; must match the artifact)
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ENCODER_BATCH_SIZE = 32
DEVICE = "cuda"                # "cuda" (Colab GPU) or "cpu"

# ---------------------------------------------------------------------------
# Dense retrieval parameters (no feedback, no fusion weights)
# ---------------------------------------------------------------------------
TOP_K_RETRIEVAL = 10
TOP_K_CONCEPT = 5               # candidates saved for e2e.ipynb
ENCODE_BATCH_SIZE = 64          # mentions encoded (and searched) per step
INDEX_KB = None                 # optional source_kb filter; None = all entries

# ---------------------------------------------------------------------------
# Evaluation / runtime
# ---------------------------------------------------------------------------
MAX_SAMPLES = None              # None = all messages; an int truncates for smoke runs
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"KB artifact  : {KB_ARTIFACT_PATH}")
print(f"Gold data    : {GOLD_DATA_PATH}")
print(f"Candidates   : {BASELINE_CANDIDATES_PATH}")
print(f"Encoder      : {ENCODER_MODEL}")

## Stage 1 — Clone or Update the Code Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/`. Re-running this cell
updates the existing clone instead of downloading it again.

In [ ]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.aggregation import aggregate_hits
from common.candidates import load_candidates, save_candidates
from common.data import load_bel_jsonl
from common.encoder import SentenceTransformerEncoder
from common.evaluator import metrics_from_responses
from common.index import SINGLE_FILE_FORMAT, VectorIndex
from common.schemas import Response

log(f"Repository ready at {REPO_CLONE_DIR}; common imported from {SOURCE_DIR}.")
log(f"Single-file artifact format: {SINGLE_FILE_FORMAT}")

## Stage 2 — Mount Google Drive and Locate Inputs

The KB artifact and the gold data live on Drive, not in the repository. This
stage mounts Drive and checks both inputs plus the single candidate output path
before any model is loaded.

In [ ]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

artifact_path = Path(require_configured(KB_ARTIFACT_PATH, "KB_ARTIFACT_PATH"))
gold_path = Path(require_configured(GOLD_DATA_PATH, "GOLD_DATA_PATH"))
candidates_path = Path(
    require_configured(BASELINE_CANDIDATES_PATH, "BASELINE_CANDIDATES_PATH")
)

for label, path in (
    ("KB artifact", artifact_path),
    ("Gold data", gold_path),
):
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    log(f"{label}: {path} ({path.stat().st_size / 1024 ** 2:.2f} MB)")

if candidates_path.suffix.lower() != ".jsonl":
    raise ValueError(
        f"BASELINE_CANDIDATES_PATH must end in .jsonl, got {candidates_path.name!r}."
    )
log(f"Candidate output: {candidates_path}")

## Stage 3 — Load the KB Artifact and the Encoder

`VectorIndex.load_npz` restores the vectors and metadata from the single file.
The encoder model itself is not stored in the artifact, so this stage loads the
same checkpoint recorded in `encoder_info` and checks the dimensions agree.
With `DEVICE = "cuda"`, `index.use_gpu()` keeps one GPU copy of the vectors:
retrieval is then a batched matrix multiply on the GPU, and only the top-k rows
cross back to CPU.

In [ ]:
log(f"Loading KB artifact {artifact_path} ...")
index = VectorIndex.load_npz(artifact_path)
log(f"Artifact loaded: {len(index):,} entries x {index.vectors.shape[1]} dims.")

stored_encoder = (index.encoder_info or {}).get("model")
if stored_encoder is None:
    log("WARNING: artifact has no encoder_info; assuming ENCODER_MODEL is the same checkpoint.")
elif stored_encoder != ENCODER_MODEL:
    raise ValueError(
        f"Artifact was built with encoder {stored_encoder!r}, but ENCODER_MODEL is "
        f"{ENCODER_MODEL!r}. Set ENCODER_MODEL to the encoder used by build_KB.ipynb."
    )

if DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "DEVICE='cuda' but no CUDA device is available. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU."
        )
    log(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    log("Running on CPU (DEVICE='cpu').")

log(f"Loading encoder {ENCODER_MODEL!r} on {DEVICE} ...")
encoder = SentenceTransformerEncoder(
    ENCODER_MODEL, batch_size=ENCODER_BATCH_SIZE, device=DEVICE
)
if encoder.dim != index.vectors.shape[1]:
    raise ValueError(
        f"Encoder dimension {encoder.dim} does not match artifact dimension "
        f"{index.vectors.shape[1]}."
    )
log(f"Encoder ready: dim={encoder.dim}.")

if DEVICE == "cuda":
    try:
        index.use_gpu(DEVICE)
    except RuntimeError as exc:
        torch.cuda.empty_cache()
        log(f"WARNING: GPU search unavailable ({exc}); using the CPU scan.")
    else:
        log(f"Search device: {index.search_device} (exact cosine on GPU).")
if index.search_device == "cpu":
    log("Search device: cpu (exact cosine on CPU).")

## Stage 4 — Load and Validate the Gold Data

`load_bel_jsonl` validates structure and token spans. Structural problems are
reported as `DataIssue` entries; span issues never drop a sample. All samples
are used for candidate retrieval. The loader reports each record through
`progress_callback`, which drives the bar.

In [ ]:
log(f"Loading gold BEL data from {gold_path} ...")
started = time.time()
with progress_bar(None, "Loading gold records", "record") as report:
    loaded = load_bel_jsonl(gold_path, strict=False, progress_callback=report)
samples = list(loaded.samples)
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]

log(
    f"Loaded {len(loaded.samples):,} mention samples "
    f"({len(samples):,} selected) in {time.time() - started:.1f}s."
)
if loaded.issues:
    log(f"WARNING: {len(loaded.issues):,} data issue(s); first: {loaded.issues[0]}")
if not samples:
    raise ValueError(f"{gold_path} produced no usable mention samples.")

## Stage 5 — Dense Retriever

The baseline encodes each mention and scans the KB vectors directly: the
Top-1 aggregated concept is the prediction. There is no feedback call, no
fusion weight and no LLM client in this path.

Mentions are processed in `ENCODE_BATCH_SIZE` chunks so the progress bar moves
during encoding/searching. `aggregate_hits` keeps the full untruncated concept
ranking for R@5 and truncates only the candidate list saved for `e2e.ipynb`.

In [ ]:
if TOP_K_RETRIEVAL < 5:
    log("WARNING: TOP_K_RETRIEVAL < 5, so R@5 cannot exceed TOP_K_RETRIEVAL.")
if not 1 <= TOP_K_CONCEPT <= 10:
    raise ValueError("TOP_K_CONCEPT must be in 1..10.")


def dense_responses(samples, report):
    """Embed mentions in batches and return one Response per sample."""
    responses = []
    for start in range(0, len(samples), ENCODE_BATCH_SIZE):
        batch = samples[start : start + ENCODE_BATCH_SIZE]
        vectors = encoder.encode([sample.mention for sample in batch])
        hit_lists = index.search_batch(
            vectors, TOP_K_RETRIEVAL, source_kb=INDEX_KB
        )
        for sample, hits in zip(batch, hit_lists):
            ranking = aggregate_hits(hits)
            if not ranking:
                raise ValueError(
                    f"No candidate for {sample.mention!r} "
                    f"(record {sample.record_index}, entity {sample.entity_index})."
                )
            candidates = tuple(ranking[:TOP_K_CONCEPT])
            responses.append(
                Response(
                    cui=candidates[0].cui,
                    candidates=candidates,
                    debug={
                        "concept_ranking": [
                            {
                                "cui": candidate.cui,
                                "score": candidate.score,
                                "surface": candidate.surface,
                            }
                            for candidate in ranking
                        ],
                        "score_kind": "embedding",
                    },
                )
            )
        report(start + len(batch), len(samples))
    return responses


log(
    f"Dense retriever ready: {len(index):,} entries; "
    f"top_k_retrieval={TOP_K_RETRIEVAL} top_k_concept={TOP_K_CONCEPT} "
    f"batch={ENCODE_BATCH_SIZE}."
)

## Stage 6 — Run Dense Retrieval

One encode-and-search pass produces, per mention, the Top-1 prediction, the
candidate list and the untruncated concept ranking. The batch loop reports each
finished chunk through the progress bar.

In [ ]:
with progress_bar(len(samples), "Dense retrieval", "mention") as report:
    started = time.time()
    responses = dense_responses(samples, report)

log(
    f"Retrieved candidates for {len(responses):,} mentions "
    f"in {time.time() - started:.1f}s."
)

## Stage 7 — Save Candidates to One File

`save_candidates` writes one JSONL record per mention to
`BASELINE_CANDIDATES_PATH` in the same format as `retrieval.ipynb`, so
`e2e.ipynb` can consume either file. The file is reloaded immediately to prove
it is usable.

In [ ]:
log(f"Saving candidates to {candidates_path} ...")
started = time.time()
with progress_bar(len(samples), "Saving candidate records", "record") as report:
    save_candidates(candidates_path, samples, responses, progress_callback=report)
log(
    f"Saved {len(responses):,} records "
    f"({candidates_path.stat().st_size / 1024 ** 2:.2f} MB) "
    f"in {time.time() - started:.1f}s."
)

log("Reloading the candidate file to verify it ...")
with progress_bar(None, "Reloading candidate records", "record") as report:
    reloaded_samples, reloaded_responses = load_candidates(
        candidates_path, progress_callback=report
    )
assert len(reloaded_samples) == len(samples), (len(reloaded_samples), len(samples))
assert len(reloaded_responses) == len(responses), (len(reloaded_responses), len(responses))
assert [sample.gold_cui for sample in reloaded_samples] == [
    sample.gold_cui for sample in samples
]
assert [response.candidates for response in reloaded_responses] == [
    response.candidates for response in responses
]
assert [
    response.debug.get("concept_ranking") or []
    for response in reloaded_responses
] == [
    response.debug.get("concept_ranking") or []
    for response in responses
]
log(f"Verified: {len(reloaded_samples):,} samples and responses reloaded.")

## Stage 8 — Baseline Metrics

R@1/R@5 come from the untruncated retrieval ranking. Accuracy is the fraction
of mentions whose Top-1 candidate equals the gold CUI — the baseline has no
further re-ranking step.

In [ ]:
overall, by_label = metrics_from_responses(samples, responses)


def _row(label, metrics) -> str:
    return (
        f"{str(label):<16}{metrics.n:>7}"
        f"{100 * metrics.recall_at_1:>10.2f}{100 * metrics.recall_at_5:>10.2f}"
        f"{100 * metrics.accuracy:>10.2f}"
    )


print(f"{'label':<16}{'n':>7}{'R@1 %':>10}{'R@5 %':>10}{'acc %':>10}")
print(_row("ALL", overall))
for label, metrics in by_label.items():
    print(_row(label, metrics))

log("Dense baseline complete. Candidate file can also feed e2e.ipynb.")

## Stage 9 — Release Memory

Candidates are saved and the metrics are printed, so drop the vector index
(including its GPU copy), the encoder and the loaded samples/responses, then
empty the CUDA cache.

In [ ]:
if "index" in globals():
    index.release_gpu()

for name in (
    "index",
    "encoder",
    "loaded",
    "samples",
    "responses",
    "reloaded_samples",
    "reloaded_responses",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("Baseline memory released (index, encoder, samples, responses).")